In [10]:
import sys
import optuna
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from shapely.geometry import Point

sys.path.append("../..")
from src.qubo_windfarm_layout.model import get_farm_area, compute_aep_from_coords
from src.qubo_windfarm_layout.evaluation import load_layout_coordinates, get_farm_area

In [11]:
def optuna_promising_turbines_refinement(
    layout_yaml,
    candidate_locations,
    wake_loss_matrix,
    step_m,
    radius_m,
    n_promising_turbines=15,
    n_trials_per_turbine=30,
    min_distance=396.0,
    max_outer_iterations=30,
    min_improvement=1e-6,
    match_tol=1e-6,
    boundary_tol_m=1e-6,
    seed=42,
    verbose=True,
):
    """
    Hybrid physics-based local refinement.

    Ad ogni outer iteration:

    1. identifica le turbine che NON giacciono sul boundary;
    2. calcola il wake interaction score solo per questo subset;
    3. seleziona le top-N turbine interne con wake score maggiore;
    4. per ogni turbina selezionata:
           - tiene ferme le altre 80 turbine;
           - genera candidate locali entro radius_m;
           - filtra boundary e minimum spacing;
           - usa Optuna per ottimizzare dx, dy;
           - massimizza il full AEP;
    5. confronta il miglior movimento trovato per ciascuna turbina;
    6. accetta UN SOLO movimento: quello con AEP maggiore;
    7. aggiorna la pairwise wake interaction matrix;
    8. ripete.

    Le turbine che giacciono sul boundary non vengono considerate
    come possibili turbine da spostare.
    """

    # ==================================================
    # INITIAL LAYOUT
    # ==================================================

    current_positions = np.asarray(
        load_layout_coordinates(layout_yaml),
        dtype=float,
    )

    candidate_locations = np.asarray(
        candidate_locations,
        dtype=float,
    )

    wake_loss_matrix = np.asarray(
        wake_loss_matrix,
        dtype=float,
    )

    n_turbines = len(current_positions)

    _, farm_area = get_farm_area()

    # --------------------------------------------------
    # Map initial YAML layout -> wake matrix indices
    # --------------------------------------------------

    candidate_tree = cKDTree(
        candidate_locations
    )

    distances, selected_indices = candidate_tree.query(
        current_positions,
        k=1,
    )

    if np.any(distances > match_tol):
        raise ValueError(
            "Il layout YAML non è compatibile con "
            "candidate_locations. "
            f"Max distance = {distances.max():.6f} m"
        )

    if len(np.unique(selected_indices)) != n_turbines:
        raise ValueError(
            "Più turbine sono state associate "
            "alla stessa candidate location."
        )

    # --------------------------------------------------
    # Interaction matrix of the current 81 turbines
    # --------------------------------------------------

    current_L = wake_loss_matrix[
        np.ix_(
            selected_indices,
            selected_indices,
        )
    ].copy()

    # --------------------------------------------------
    # Single turbine AEP
    # --------------------------------------------------

    A0 = float(
        compute_aep_from_coords(
            [current_positions[0]]
        )
    )

    current_aep = float(
        compute_aep_from_coords(
            current_positions
        )
    )

    initial_aep = current_aep

    history = []
    all_trials = []

    # ==================================================
    # LOCAL GRID
    # ==================================================

    n_steps = int(
        np.floor(radius_m / step_m)
    )

    offsets = (
        np.arange(
            -n_steps,
            n_steps + 1,
        )
        * step_m
    )

    dx_grid, dy_grid = np.meshgrid(
        offsets,
        offsets,
    )

    local_offsets = np.column_stack(
        (
            dx_grid.ravel(),
            dy_grid.ravel(),
        )
    )

    # Circular neighbourhood
    local_offsets = local_offsets[
        np.linalg.norm(
            local_offsets,
            axis=1,
        )
        <= radius_m + 1e-9
    ]

    # Remove zero displacement
    local_offsets = local_offsets[
        np.linalg.norm(
            local_offsets,
            axis=1,
        )
        > 1e-9
    ]

    # ==================================================
    # INITIAL LOG
    # ==================================================

    if verbose:
        print("=" * 70)
        print("OPTUNA INTERIOR-TURBINE REFINEMENT")
        print("=" * 70)

        print(
            f"Turbines:               "
            f"{n_turbines}"
        )

        print(
            f"Step:                   "
            f"{step_m} m"
        )

        print(
            f"Radius:                 "
            f"{radius_m} m"
        )

        print(
            f"Boundary tolerance:     "
            f"{boundary_tol_m} m"
        )

        print(
            f"Local grid positions:   "
            f"{len(local_offsets)}"
        )

        print(
            f"Promising turbines:     "
            f"{n_promising_turbines}"
        )

        print(
            f"Trials / turbine:       "
            f"{n_trials_per_turbine}"
        )

        print(
            f"Initial AEP:            "
            f"{current_aep:.6f}"
        )

    # ==================================================
    # OUTER LOOP
    # ==================================================

    for outer_iteration in range(
        max_outer_iterations
    ):

        if verbose:
            print()
            print("=" * 70)
            print(
                f"OUTER ITERATION "
                f"{outer_iteration + 1}/"
                f"{max_outer_iterations}"
            )
            print("=" * 70)

            print(
                f"Current AEP: "
                f"{current_aep:.6f}"
            )

        # ==================================================
        # 1. WAKE SCORE
        # ==================================================

        wake_scores = current_L.sum(
            axis=1
        )

        # ==================================================
        # 2. IDENTIFY BOUNDARY TURBINES
        # ==================================================

        on_boundary = np.array(
            [
                farm_area.boundary.distance(
                    Point(x, y)
                )
                <= boundary_tol_m

                for x, y in current_positions
            ],
            dtype=bool,
        )

        interior_indices = np.flatnonzero(
            ~on_boundary
        )

        if verbose:
            print(
                f"Boundary turbines:      "
                f"{on_boundary.sum()}"
            )

            print(
                f"Interior turbines:      "
                f"{len(interior_indices)}"
            )

        if len(interior_indices) == 0:

            if verbose:
                print(
                    "Tutte le turbine giacciono "
                    "sul boundary."
                )

            break

        # ==================================================
        # 3. RANK ONLY INTERIOR TURBINES
        # ==================================================

        interior_scores = wake_scores[
            interior_indices
        ]

        order = np.argsort(
            interior_scores
        )[::-1]

        n_select = min(
            n_promising_turbines,
            len(interior_indices),
        )

        promising_turbines = (
            interior_indices[
                order[:n_select]
            ]
        )

        if verbose:
            print(
                f"Selected turbines:      "
                f"{len(promising_turbines)}"
            )

            print()
            print(
                "Most promising interior turbines:"
            )

            for rank, i in enumerate(
                promising_turbines,
                start=1,
            ):
                print(
                    f"  {rank:02d}. "
                    f"T{i:02d} | "
                    f"wake score="
                    f"{wake_scores[i]:.6f}"
                )

        # ==================================================
        # 4. OPTIMIZE EACH PROMISING TURBINE
        # ==================================================

        turbine_results = []

        for turbine_rank, turbine_idx in enumerate(
            promising_turbines,
            start=1,
        ):

            turbine_idx = int(
                turbine_idx
            )

            origin = current_positions[
                turbine_idx
            ].copy()

            others = np.delete(
                current_positions,
                turbine_idx,
                axis=0,
            )

            # ------------------------------------------
            # Generate local candidates
            # ------------------------------------------

            local_candidates = (
                origin
                + local_offsets
            )

            # ------------------------------------------
            # Farm boundary filter
            # ------------------------------------------

            boundary_mask = np.array(
                [
                    farm_area.covers(
                        Point(x, y)
                    )
                    for x, y in local_candidates
                ]
            )

            local_candidates = (
                local_candidates[
                    boundary_mask
                ]
            )

            if len(local_candidates) == 0:
                continue

            # ------------------------------------------
            # Minimum spacing filter
            # ------------------------------------------

            feasible_candidates = []

            for candidate in local_candidates:

                distances_to_others = (
                    np.linalg.norm(
                        others - candidate,
                        axis=1,
                    )
                )

                if np.all(
                    distances_to_others
                    >= min_distance
                ):
                    feasible_candidates.append(
                        candidate
                    )

            if not feasible_candidates:
                continue

            feasible_candidates = np.asarray(
                feasible_candidates,
                dtype=float,
            )

            feasible_candidates = np.unique(
                np.round(
                    feasible_candidates,
                    6,
                ),
                axis=0,
            )

            feasible_offsets = (
                feasible_candidates
                - origin
            )

            feasible_tree = cKDTree(
                feasible_offsets
            )

            if verbose:
                print()
                print(
                    f"Turbine {turbine_idx:02d} "
                    f"({turbine_rank}/"
                    f"{len(promising_turbines)})"
                )

                print(
                    f"  Wake score:     "
                    f"{wake_scores[turbine_idx]:.6f}"
                )

                print(
                    f"  Feasible moves: "
                    f"{len(feasible_candidates)}"
                )

            # ==========================================
            # OPTUNA STUDY FOR ONE FIXED TURBINE
            # ==========================================

            cache = {}

            def objective(trial):

                # --------------------------------------
                # Optuna proposes dx / dy
                # --------------------------------------

                proposed_dx = trial.suggest_float(
                    "dx",
                    -radius_m,
                    radius_m,
                    step=step_m,
                )

                proposed_dy = trial.suggest_float(
                    "dy",
                    -radius_m,
                    radius_m,
                    step=step_m,
                )

                proposed_offset = np.array(
                    [
                        proposed_dx,
                        proposed_dy,
                    ],
                    dtype=float,
                )

                # --------------------------------------
                # Snap to nearest FEASIBLE candidate
                # --------------------------------------

                _, candidate_idx = (
                    feasible_tree.query(
                        proposed_offset,
                        k=1,
                    )
                )

                candidate_idx = int(
                    candidate_idx
                )

                candidate = (
                    feasible_candidates[
                        candidate_idx
                    ]
                )

                actual_offset = (
                    candidate
                    - origin
                )

                dx = float(
                    actual_offset[0]
                )

                dy = float(
                    actual_offset[1]
                )

                move_distance = float(
                    np.linalg.norm(
                        actual_offset
                    )
                )

                # --------------------------------------
                # Cache
                # --------------------------------------

                key = (
                    round(
                        float(candidate[0]),
                        6,
                    ),
                    round(
                        float(candidate[1]),
                        6,
                    ),
                )

                if key in cache:

                    aep = cache[key]

                else:

                    trial_positions = (
                        current_positions.copy()
                    )

                    trial_positions[
                        turbine_idx
                    ] = candidate

                    aep = float(
                        compute_aep_from_coords(
                            trial_positions
                        )
                    )

                    cache[key] = aep

                delta_aep = (
                    aep - current_aep
                )

                # --------------------------------------
                # Trial metadata
                # --------------------------------------

                trial.set_user_attr(
                    "turbine_idx",
                    turbine_idx,
                )

                trial.set_user_attr(
                    "candidate_idx",
                    candidate_idx,
                )

                trial.set_user_attr(
                    "actual_dx",
                    dx,
                )

                trial.set_user_attr(
                    "actual_dy",
                    dy,
                )

                trial.set_user_attr(
                    "distance_m",
                    move_distance,
                )

                trial.set_user_attr(
                    "candidate_x",
                    float(candidate[0]),
                )

                trial.set_user_attr(
                    "candidate_y",
                    float(candidate[1]),
                )

                trial.set_user_attr(
                    "delta_aep",
                    float(delta_aep),
                )

                if verbose:
                    print(
                        f"    Trial "
                        f"{trial.number + 1:03d} | "
                        f"dx={dx:+6.1f} "
                        f"dy={dy:+6.1f} | "
                        f"d={move_distance:6.2f} m | "
                        f"AEP={aep:.6f} | "
                        f"ΔAEP="
                        f"{delta_aep:+.6f}"
                    )

                return aep

            # ==========================================
            # OPTUNA SAMPLER
            # ==========================================

            sampler = (
                optuna.samplers.TPESampler(
                    seed=(
                        seed
                        + outer_iteration * 1000
                        + turbine_idx
                    ),
                    multivariate=True,
                )
            )

            study = optuna.create_study(
                direction="maximize",
                sampler=sampler,
            )

            study.optimize(
                objective,
                n_trials=n_trials_per_turbine,
                show_progress_bar=False,
            )

            # ------------------------------------------
            # Store trials
            # ------------------------------------------

            df_trials = (
                study.trials_dataframe()
            )

            df_trials.insert(
                0,
                "outer_iteration",
                outer_iteration + 1,
            )

            df_trials.insert(
                1,
                "turbine_idx",
                turbine_idx,
            )

            all_trials.append(
                df_trials
            )

            # ------------------------------------------
            # Best move for this turbine
            # ------------------------------------------

            best_trial = (
                study.best_trial
            )

            best_candidate = np.array(
                [
                    best_trial.user_attrs[
                        "candidate_x"
                    ],
                    best_trial.user_attrs[
                        "candidate_y"
                    ],
                ],
                dtype=float,
            )

            turbine_best_aep = float(
                study.best_value
            )

            turbine_delta_aep = (
                turbine_best_aep
                - current_aep
            )

            turbine_results.append(
                {
                    "turbine_idx":
                        turbine_idx,

                    "wake_score":
                        float(
                            wake_scores[
                                turbine_idx
                            ]
                        ),

                    "from_position":
                        origin,

                    "to_position":
                        best_candidate,

                    "dx":
                        float(
                            best_candidate[0]
                            - origin[0]
                        ),

                    "dy":
                        float(
                            best_candidate[1]
                            - origin[1]
                        ),

                    "aep":
                        turbine_best_aep,

                    "delta_aep":
                        float(
                            turbine_delta_aep
                        ),

                    "n_feasible_moves":
                        len(
                            feasible_candidates
                        ),

                    "n_unique_aep_evaluations":
                        len(cache),
                }
            )

            if verbose:
                print(
                    f"  Best ΔAEP for T"
                    f"{turbine_idx:02d}: "
                    f"{turbine_delta_aep:+.6f}"
                )

        # ==================================================
        # 5. GLOBAL BEST MOVE
        # ==================================================

        if not turbine_results:

            if verbose:
                print(
                    "\nNo feasible turbine moves."
                )

            break

        best_move = max(
            turbine_results,
            key=lambda x: x["aep"],
        )

        best_aep = best_move[
            "aep"
        ]

        improvement = (
            best_aep
            - current_aep
        )

        if verbose:
            print()
            print("-" * 70)
            print("GLOBAL BEST MOVE")
            print("-" * 70)

            print(
                f"Turbine:       "
                f"{best_move['turbine_idx']}"
            )

            print(
                f"Wake score:    "
                f"{best_move['wake_score']:.6f}"
            )

            print(
                f"From:          "
                f"{best_move['from_position']}"
            )

            print(
                f"To:            "
                f"{best_move['to_position']}"
            )

            print(
                f"dx:            "
                f"{best_move['dx']:+.3f} m"
            )

            print(
                f"dy:            "
                f"{best_move['dy']:+.3f} m"
            )

            print(
                f"AEP before:    "
                f"{current_aep:.6f}"
            )

            print(
                f"AEP after:     "
                f"{best_aep:.6f}"
            )

            print(
                f"ΔAEP:          "
                f"{improvement:+.6f}"
            )

        # ==================================================
        # 6. CONVERGENCE
        # ==================================================

        if improvement <= min_improvement:

            if verbose:
                print(
                    "\nNo improving move found. "
                    "Refinement converged."
                )

            break

        # ==================================================
        # 7. ACCEPT GLOBAL BEST MOVE
        # ==================================================

        moved_idx = int(
            best_move[
                "turbine_idx"
            ]
        )

        old_position = (
            current_positions[
                moved_idx
            ].copy()
        )

        new_position = (
            best_move[
                "to_position"
            ].copy()
        )

        current_positions[
            moved_idx
        ] = new_position

        # ==================================================
        # 8. UPDATE MOVED TURBINE'S PAIRWISE INTERACTIONS
        # ==================================================

        current_L[
            moved_idx,
            :
        ] = 0.0

        current_L[
            :,
            moved_idx
        ] = 0.0

        for j in range(
            n_turbines
        ):

            if j == moved_idx:
                continue

            pair_positions = np.asarray(
                [
                    new_position,
                    current_positions[j],
                ]
            )

            pair_aep = float(
                compute_aep_from_coords(
                    pair_positions
                )
            )

            pair_loss = max(
                0.0,
                2.0 * A0
                - pair_aep,
            )

            current_L[
                moved_idx,
                j
            ] = pair_loss

            current_L[
                j,
                moved_idx
            ] = pair_loss

        # ==================================================
        # 9. HISTORY
        # ==================================================

        history.append(
            {
                "outer_iteration":
                    outer_iteration + 1,

                "turbine_idx":
                    moved_idx,

                "wake_score":
                    best_move[
                        "wake_score"
                    ],

                "from_x":
                    float(
                        old_position[0]
                    ),

                "from_y":
                    float(
                        old_position[1]
                    ),

                "to_x":
                    float(
                        new_position[0]
                    ),

                "to_y":
                    float(
                        new_position[1]
                    ),

                "dx":
                    best_move["dx"],

                "dy":
                    best_move["dy"],

                "aep_before":
                    float(
                        current_aep
                    ),

                "aep_after":
                    float(
                        best_aep
                    ),

                "delta_aep":
                    float(
                        improvement
                    ),

                "was_boundary_before":
                    bool(
                        on_boundary[
                            moved_idx
                        ]
                    ),
            }
        )

        current_aep = best_aep

    # ==================================================
    # OUTPUT
    # ==================================================

    history_df = pd.DataFrame(
        history
    )

    if all_trials:

        trials_df = pd.concat(
            all_trials,
            ignore_index=True,
        )

    else:

        trials_df = pd.DataFrame()

    if verbose:
        print()
        print("=" * 70)
        print("REFINEMENT COMPLETED")
        print("=" * 70)

        print(
            f"Initial AEP:    "
            f"{initial_aep:.6f}"
        )

        print(
            f"Final AEP:      "
            f"{current_aep:.6f}"
        )

        print(
            f"Improvement:    "
            f"{current_aep - initial_aep:+.6f}"
        )

        print(
            f"Accepted moves: "
            f"{len(history)}"
        )

    return {
        "selected_locations":
            current_positions,

        "initial_aep":
            float(initial_aep),

        "final_aep":
            float(current_aep),

        "improvement_aep":
            float(
                current_aep
                - initial_aep
            ),

        "n_swaps":
            len(history),

        "wake_interaction_matrix":
            current_L,

        "history_df":
            history_df,

        "trials_df":
            trials_df,
    }


def build_local_candidates(
    gurobi_positions,
    farm_area,
    radius_m,
    step_m,
    min_distance=396.0,
):
    gurobi_positions = np.asarray(gurobi_positions, dtype=float)

    # offset locali
    offsets_1d = np.arange(
        -radius_m,
        radius_m + step_m * 0.5,
        step_m,
    )

    dx, dy = np.meshgrid(
        offsets_1d,
        offsets_1d,
        indexing="xy",
    )

    offsets = np.column_stack([
        dx.ravel(),
        dy.ravel(),
    ])

    # tengo solo punti dentro il raggio
    offsets = offsets[
        np.linalg.norm(offsets, axis=1) <= radius_m
    ]

    # genero candidati attorno a ogni turbina Gurobi
    candidate_locations = (
        gurobi_positions[:, None, :]
        + offsets[None, :, :]
    ).reshape(-1, 2)

    # aggiungo esplicitamente le posizioni Gurobi originali
    candidate_locations = np.vstack([
        candidate_locations,
        gurobi_positions,
    ])

    # tengo solo punti dentro il farm boundary
    mask = np.array([
        farm_area.covers(Point(x, y))
        for x, y in candidate_locations
    ])

    candidate_locations = candidate_locations[mask]

    # rimuovo duplicati
    candidate_locations = np.unique(
        np.round(candidate_locations, 6),
        axis=0,
    )

    # invalid pairs
    tree = cKDTree(candidate_locations)

    invalid_pairs = tree.query_pairs(
        r=min_distance,
        output_type="ndarray",
    )

    # query_pairs include anche distanza == min_distance:
    # noi vogliamo solo < min_distance
    if len(invalid_pairs) > 0:
        d = np.linalg.norm(
            candidate_locations[invalid_pairs[:, 0]]
            - candidate_locations[invalid_pairs[:, 1]],
            axis=1,
        )

        invalid_pairs = invalid_pairs[
            d < min_distance
        ]

    return (
        candidate_locations,
        invalid_pairs.astype(np.int32),
    )

In [6]:
# Variables
LAST_STEP = 2
FINAL_STEP = 1

# Parameters
LAST_RADIUS = 2*LAST_STEP
FINAL_RADIUS = 64 #LAST_RADIUS /2

WAKE_LOSS_PATH = f"../../results/precomputed/gurobi/wake_loss_gurobi_res_{LAST_STEP}m_rad_{LAST_RADIUS}.npz"
STARTING_LAYOUT = F"../../results/layouts/gurobi/gurobi_local_{LAST_STEP}m_1800s_warmstart.yaml"

wake_loss_matrix = np.load(WAKE_LOSS_PATH)["wake_loss_matrix"]

In [13]:
gurobi_positions = load_layout_coordinates(STARTING_LAYOUT)

_, farm_area = get_farm_area()

candidate_locations, invalid_pairs = build_local_candidates(
    gurobi_positions=gurobi_positions,
    farm_area=farm_area,
    radius_m=LAST_RADIUS,
    step_m=LAST_STEP,
)



result = optuna_promising_turbines_refinement(
    layout_yaml=STARTING_LAYOUT,
    candidate_locations=candidate_locations,
    wake_loss_matrix=wake_loss_matrix,
    step_m=FINAL_STEP,
    radius_m=FINAL_RADIUS,
    n_promising_turbines=10,
    n_trials_per_turbine=20,
    max_outer_iterations=5,
    boundary_tol_m=1e-6,
    verbose=True,
)

[I 2026-10-02 14:45:55,092] A new study created in memory with name: no-name-015b8370-66c1-47e3-a40d-a24f8cde54e8


OPTUNA INTERIOR-TURBINE REFINEMENT
Turbines:               81
Step:                   1 m
Radius:                 64 m
Boundary tolerance:     1e-06 m
Local grid positions:   12852
Promising turbines:     10
Trials / turbine:       20
Initial AEP:            2901707.988550

OUTER ITERATION 1/5
Current AEP: 2901707.988550
Boundary turbines:      0
Interior turbines:      81
Selected turbines:      10

Most promising interior turbines:
  01. T21 | wake score=66258.687674
  02. T22 | wake score=66257.793691
  03. T33 | wake score=66205.469406
  04. T32 | wake score=66202.951434
  05. T15 | wake score=65943.078300
  06. T14 | wake score=65941.972526
  07. T57 | wake score=65332.357997
  08. T58 | wake score=65330.089012
  09. T61 | wake score=64937.966544
  10. T62 | wake score=64927.554159

Turbine 21 (1/10)
  Wake score:     66258.687674
  Feasible moves: 6459


[I 2026-10-02 14:46:07,184] Trial 0 finished with value: 2901682.4819738828 and parameters: {'dx': 7.0, 'dy': -13.0}. Best is trial 0 with value: 2901682.4819738828.


    Trial 001 | dx=  +7.0 dy= -13.0 | d= 14.76 m | AEP=2901682.481974 | ΔAEP=-25.506576


[I 2026-10-02 14:46:18,951] Trial 1 finished with value: 2901636.2661486417 and parameters: {'dx': -58.0, 'dy': 3.0}. Best is trial 0 with value: 2901682.4819738828.


...

    Trial 017 | dx= +49.0 dy= +16.0 | d= 51.55 m | AEP=2902157.231325 | ΔAEP=+27.071220


[I 2026-10-02 18:00:40,879] Trial 17 finished with value: 2901984.014043834 and parameters: {'dx': -3.0, 'dy': -24.0}. Best is trial 14 with value: 2902157.9405720583.


    Trial 018 | dx=  -3.0 dy= -24.0 | d= 24.19 m | AEP=2901984.014044 | ΔAEP=-146.146061


[I 2026-10-02 18:00:53,918] Trial 18 finished with value: 2902152.1584101114 and parameters: {'dx': 21.0, 'dy': 7.0}. Best is trial 14 with value: 2902157.9405720583.
[I 2026-10-02 18:00:53,922] Trial 19 finished with value: 2902148.967524116 and parameters: {'dx': 61.0, 'dy': 27.0}. Best is trial 14 with value: 2902157.9405720583.


    Trial 019 | dx= +21.0 dy=  +7.0 | d= 22.14 m | AEP=2902152.158410 | ΔAEP=+21.998305
    Trial 020 | dx= +61.0 dy= +19.0 | d= 63.89 m | AEP=2902148.967524 | ΔAEP=+18.807419
  Best ΔAEP for T36: +27.780467

----------------------------------------------------------------------
GLOBAL BEST MOVE
----------------------------------------------------------------------
Turbine:       62
Wake score:    66054.357835
From:          [8593.4 1984.9]
To:            [8550.4 2031.9]
dx:            -43.000 m
dy:            +47.000 m
AEP before:    2902130.160105
AEP after:     2902202.264464
ΔAEP:          +72.104359

REFINEMENT COMPLETED
Initial AEP:    2901707.988550
Final AEP:      2902202.264464
Improvement:    +494.275915
Accepted moves: 5


In [ ]:
result.keys()

In [8]:
result['initial_aep']

2901707.9885497442

In [9]:
result['final_aep']

2901819.1544724926